In [ ]:
import random
from datasets import load_dataset
from tqdm import tqdm

# =============================================================================
# 🌟 코딩 튜터의 설명: KoLIMA 데이터셋 탐험하기 (AI 실습 예제) 🌟
# 🚀 데이터셋 이름: taeshahn/ko-lima
# ✨ 의미: KoLIMA는 Meta의 LIMA(Less Is More for Alignment) 학습 데이터를 
#       한국어로 번역한 대화형(Instruction/Conversation) 데이터셋입니다.
# 📖 용도: 이 데이터셋은 AI 모델이 사용자의 질문(Source)에 맞춰 
#       매우 자연스럽고 일관된 대화 흐름(conversations)을 이어가도록 
#       학습하는 데 사용됩니다.
# 🎯 실습 목표: 이 코드를 통해 '대화'라는 복잡한 구조를 어떻게 
#            분해하고, AI 모델이 학습할 수 있는 깨끗한 '프롬프트' 형태로 
#            재구성하는지 원리를 이해해봅시다! (입문자도 완벽 이해 가능!)
# =============================================================================

# 사용 가능한 Config 이름을 먼저 확인해요! (매우 중요합니다!)
DATASET_NAME = "taeshahn/ko-lima"
try:
    configs = get_dataset_config_names(DATASET_NAME)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    # 일반적으로 가장 기본인 'plain'이나 첫 번째 Config를 사용해요.
    selected_config = configs[0]
except Exception as e:
    print(f"ℹ️ Config 목록 확인 중 오류 발생: {e}. 기본 설정을 시도합니다.")
    selected_config = None


# --- [설정 변수] ---
# 📚 실습할 샘플 개수 (데이터 전체를 로드하지 않도록, 소량만 샘플링합니다!)
SAMPLE_COUNT = 5
print(f"\n==========================================================")
print(f"✨ [{DATASET_NAME}] 데이터셋을 {SAMPLE_COUNT}개 샘플로 분석합니다.")
print(f"==========================================================\n")

# -----------------------------------------------------------------------------
# 💡 1단계: 데이터 로드 마스터하기 (스트리밍 vs. 일반 로드)
# -----------------------------------------------------------------------------
# 스트리밍 로드는 메모리 효율이 좋지만, 일부 함수에서 오류가 날 수 있어요.
# 그래서 'try-except' 구문을 사용해 안전하게 로드 방법을 결정합니다.

full_dataset = None

print("🔎 1단계: 데이터 로드를 시도합니다... (스트리밍 우선)")
try:
    # 🌐 우선 스트리밍 모드(streaming=True)로 로드 시도
    # 메모리를 적게 쓰면서 데이터를 하나씩 처리하는 최신 기법입니다!
    full_dataset = load_dataset(DATASET_NAME, name=selected_config, split='train', streaming=True)
    print("🎉 성공! 스트리밍 데이터셋으로 로드에 성공했습니다. (메모리 절약 모드!)")
    
except Exception as e:
    # 🚨 스트리밍 로드 실패 시 (예: 일부 환경 제약사항)
    print(f"⚠️ 스트리밍 로드에 실패했습니다 ({type(e).__name__}). 일반 로드 모드로 전환합니다.")
    try:
        # 💾 일반 로드 모드 (적은 샘플만 로드하여 메모리 부담 최소화)
        full_dataset = load_dataset(DATASET_NAME, name=selected_config, split='train')
        print("✅ 성공! 일반 데이터셋으로 로드에 성공했습니다.")
    except Exception as e_fallback:
        print(f"❌ 치명적 오류: 데이터셋 로드에 실패했습니다. 확인해주세요. ({e_fallback})")
        exit()

# -----------------------------------------------------------------------------
# 🧪 2단계: 안전한 샘플링 및 데이터 확보 (가장 중요한 기술!)
# -----------------------------------------------------------------------------
# streaming 데이터셋은 len()을 사용할 수 없기 때문에, .take()와 list() 패턴을 사용합니다.
if hasattr(full_dataset, "take"):
    # 스트리밍 데이터셋 패턴 사용 (IterableDataset)
    print(f"\n➡️ {SAMPLE_COUNT}개의 샘플을 Iterator로 추출합니다.")
    sampled_dataset_iterator = full_dataset.take(SAMPLE_COUNT)
    # 리스트로 변환하여 나중에 반복 작업을 쉽게 하도록 만듭니다.
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 데이터셋 패턴 사용 (Dataset)
    print(f"\n➡️ {SAMPLE_COUNT}개의 샘플을 직접 추출합니다.")
    # 데이터셋 전체가 너무 클 수 있으니, 리스트로 변환하여 사용합니다.
    sample_data_list = list(full_dataset.select(range(min(SAMPLE_COUNT, len(full_dataset)))))


# -----------------------------------------------------------------------------
# 💡 3단계: 핵심 로직 - 대화 구조 분석 및 프롬프트 재구성
# -----------------------------------------------------------------------------
def format_conversation_to_prompt(conversation_list):
    """
    대화형 리스트(Role: Text)를 AI 모델 학습에 적합한 단일 프롬프트 문자열로 변환합니다.
    (예: User: Q? \n Assistant: A!)
    """
    prompt = ""
    for i, turn in enumerate(conversation_list):
        # 대화 턴이 list 형태로 주어지므로, 순회하며 내용을 추출합니다.
        role = turn['role']
        content = turn['content']
        
        if i == 0:
            # 첫 번째 턴 (최상위 질문)은 시작점처럼 강조할 수 있어요.
            prompt += f"사용자 요청: {content}\n"
        elif role == 'user':
            # 사용자의 추가 발언 (Follow-up question 등)
            prompt += f"사용자: {content}\n"
        elif role == 'assistant':
            # AI의 답변 (모델이 생성해야 할 부분, 학습 목표)
            # 여기에 `<|im_end|>` 같은 토크나이저 구분자를 넣기도 합니다.
            prompt += f"AI 답변: {content}\n"
    return prompt.strip()


# -----------------------------------------------------------------------------
# 🎁 4단계: 실습 시작! (반복문과 창의적 데이터 분석)
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print("🧠 실습 시작: 데이터 분석 및 AI 프롬프트 생성 🤖")
print("="*70)

# 분석 결과를 저장할 리스트
analyzed_prompts = []
total_turns = 0

# 샘플 데이터를 하나씩 꺼내서 분석을 시작합니다.
for i, sample in enumerate(tqdm(sample_data_list, desc="🔍 샘플 분석 중")):
    
    # 🔑 데이터 구조 접근 (conversations 필드에 리스트가 들어있습니다.)
    conversations = sample['conversations']
    
    # 1. 대화 턴 개수 분석 (Quantitative Analysis)
    current_turn_count = len(conversations)
    total_turns += current_turn_count
    
    # 2. 프롬프트 재구성 (핵심 AI 로직)
    formatted_prompt = format_conversation_to_prompt(conversations)
    
    # 3. 결과 저장
    analyzed_prompts.append({
        'sample_index': i + 1,
        'turns': current_turn_count,
        'prompt': formatted_prompt
    })

# -----------------------------------------------------------------------------
# 📊 5단계: 분석 결과 보고서 (튜터 코멘트 스타일)
# -----------------------------------------------------------------------------
print("\n" + "#"*70)
print("📚 데이터 분석 결과 보고서 (Report Card)")
print("#"*70)

# A. 전체 통계 분석
average_turns = total_turns / len(sample_data_list) if sample_data_list else 0
print(f"📊 [요약 통계] 분석된 샘플 개수: {len(sample_data_list)}개")
print(f"📊 [요약 통계] 총 분석된 대화 턴(Turn) 수: {total_turns}개")
print(f"🎯 [결론] 평균 대화 턴 수: {average_turns:.2f}개 (꽤 깊이 있는 대화들이네요!)")
print("-" * 70)


# B. 샘플별 상세 분석 출력
for sample_data in analyzed_prompts:
    print(f"💡 [샘플 {sample_data['sample_index']}] 대화 턴 수: {sample_data['turns']}개")
    print("--- 프롬프트 내용 ---")
    # AI가 모델 학습에 사용하는 최종 출력 형태를 보여줍니다.
    print(sample_data['prompt'])
    print("-" * 30)